In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
import re
import time
import random
import matplotlib.pyplot as plt
import math
from sklearn.linear_model import LinearRegression
import numpy as np

In [ ]:
receipts = pd.read_csv('sw_casestudy_receipts.csv')

# 1. Web Scraping

I created a function named `scrape_brands` to collect all the brand names from the Mondelez website using Selenium WebDriver. I then run this function repeatedly in a loop until it successfully gathers all 42 brands.


In [ ]:
def scrape_brands():
    # Set Chrome options for headless browsing
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Run in headless mode

    # Initialize WebDriver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

    # Target URL
    url = "https://www.mondelezinternational.com/our-brands/"
    driver.get(url)
    brands = []

    # Use a for loop to find all elements
    for i in range(1, 43):  
        xpath = f'/html/body/div/div[1]/div/div[2]/main/div[5]/div[{i}]/div/a'
        try:
            brand_element = driver.find_element(By.XPATH, xpath)
            brands.append(brand_element.text)
        except Exception as e:
            print(f"No element found at div[{i}]. Stopping.")
            break

    driver.quit()
    
    return brands

# Retry loop to ensure we collect all 42 brands
attempt = 0
while True:
    attempt += 1
    print(f"Attempt {attempt}: Scraping brands...")
    brands = scrape_brands()

    # Check if we got exactly 42 brands
    if len(brands) == 42:
        print(f"Successfully found all 42 brands on attempt {attempt}.")
        break
    else:
        print(f"Only found {len(brands)} brands")

    time.sleep(10)  

# Print the collected brand names
print("Final list of brands:")
for brand in brands:
    print(brand)

# 2. Matching Brands

Rows are matched to brands through a 2-step process, The steps are as follows:

- Capture Brand Comparison: The `capture_brand column` is compared directly against the list of brands, looking for an exact match.
- Item Description Parsing: Next, the `item_description` column is checked to see if it contains the brand name or a variation of the brand.

In [ ]:
# Define the brand variations dictionary (subset of the brands)
brand_variations = {
    'CHIPS AHOY!': ['chips ahoy', 'chips ahoy!'],  # Handle both with and without exclamation mark
    'SOUR PATCH KIDS': ['sour patch kids', 'sour patch'],  # Handle partial match
    'MAYNARDS BASSETT\'S': ['maynards bassetts', 'swedish fish', 'Bassett\'s'],
    'TATE\'S BAKE SHOP': ['tate\'s', 'tate\'s bake shop', 'tate\'s natural miracle', 'suzanne tate\'s nature series']
}

# Function to build regex with word boundaries for each brand
def build_word_boundary_regex(brand):
    return re.compile(rf'(^|\s|\W){re.escape(brand)}(\s|\W|$)', re.IGNORECASE)

# Compile the regular expressions for brands in the brands list (without variations)
brand_patterns = {brand: build_word_boundary_regex(brand) for brand in brands}

# Add patterns for the brands with variations from brand_variations
variation_patterns = {brand: [build_word_boundary_regex(variation) for variation in variations] for brand, variations in brand_variations.items()}

# Step 1: Try to match based on the 'capture_brand' column
def match_brand_from_capture(row):
    matched_brands = []
    if pd.isnull(row['capture_brand']):
        return matched_brands
    for brand, pattern in brand_patterns.items():
        if pattern.search(row['capture_brand']):
            matched_brands.append(brand)
    return matched_brands

# Step 2: Match based on the 'item_description' column
def match_brand_from_description(row):
    matched_brands = []
    for brand, pattern in brand_patterns.items():
        if pattern.search(row['item_description']):
            matched_brands.append(brand)
    return matched_brands

# Step 3: Match based on the brand variations
def match_brand_from_variations(row):
    matched_brands = []
    for brand, patterns in variation_patterns.items():
        for pattern in patterns:
            if pattern.search(row['item_description']):
                matched_brands.append(brand)
                break  # Break to avoid appending the same brand multiple times for different variations
    return matched_brands

# Step 4: Combine all matching functions to get a list of matched brands
def find_all_matches(row):
    matched_brands = set() 
    
    matched_brands.update(match_brand_from_capture(row))
    
    matched_brands.update(match_brand_from_description(row))
    
    matched_brands.update(match_brand_from_variations(row))
    
    return list(matched_brands)

# Step 5: Randomly assign one brand from the list of matched brands (if any)
def assign_random_brand(row):
    matched_brands = find_all_matches(row)
    if matched_brands:
        return random.choice(matched_brands)  
    return None  

# Apply the matching function to each row and store the result in 'matched_brand' as a string
receipts['matched_brand'] = receipts.apply(assign_random_brand, axis=1)

# 3. Item Categoriziation 

### Methodology: 
- Assigning Categories: I used the capture_category to categorize the different purchases. My code fills the missing values in the capture_category using a two-step process:
    - Category Mapping: I begin by mapping the item_description and barcode_category_3 to their corresponding capture_category values. If a purchase includes the same item_description or  barcode_category_3 as a row that already has a filled capture_category, it is assigned the same capture_category as that match. This approach is based on the assumption that items sharing the same item_description or barcode_category_3 are likely to belong to the same category.
    - Majority Category Mapping: For the remaining rows with missing capture_category, the most frequent capture_category associated with each matched_brand is calculated. This majority category is then used to fill missing values, based on the expectation that brands are generally associated with a consistent product category

- Data Cleaning: Categories in the capture_category column with insufficient data or non-specific descriptions are removed to ensure the analysis is focused on relevant and accurate information. To remove the impact of low sales volumes that could skew the analysis, the first and last two months of data are excluded from the calculation.
- Sales Growth Calculation: I calculate the 6-Month Sales Growth (comparing the previous 6 months to the 6 months prior) and the 12-Month Sales Growth (comparing the previous 12 months to the 12 months prior) for each category. By averaging over multiple months, this method helps to mitigate the impact of short-term fluctuations, providing a more stable and reliable view of growth trends over time.
- Trend Line Fitting: A trend line is fitted to visualize long-term growth. The coefficient of this trend line is scaled to account for variations in sales volume across different categories, ensuring the trend line reflects actual growth trends.
- Visualization: Monthly sales for each category are then visualized with the trend line superimposed.



In [ ]:
# Create a mapping (dictionary) of item descriptions to their 'capture_category' values
# Fill missing 'capture_category' values in the 'receipts' DataFrame using the category_map dict
category_map = receipts.dropna(subset=['capture_category']).set_index('item_description')['capture_category'].to_dict()
receipts.loc[:, 'capture_category'] = receipts['capture_category'].fillna(receipts['item_description'].map(category_map))

# Step 2: Create another mapping using 'barcode_category_3' as the key and 'capture_category' as the value
# Fill missing 'capture_category' values in the 'receipts' DataFrame using the category_map_3 dict
category_map_3 = receipts.dropna(subset=['barcode_category_3']).set_index('barcode_category_3')['capture_category'].to_dict()
receipts.loc[:,'capture_category'] = receipts['capture_category'].fillna(receipts['barcode_category_3'].map(category_map_3))

#Filter the 'receipts' DataFrame to keep rows where 'capture_category' or 'matched_brand' is not missing. 
# This is done to avoid having a majority vote for the null matched_brands=
receipts = receipts[(~receipts['capture_category'].isna()) | (~receipts['matched_brand'].isna()) ]

# Assigns a majority capture_category for each brand.
# Use this most common category to assign still null capture_categores
majority_category = receipts.dropna(subset=['capture_category']) \
                           .groupby('matched_brand')['capture_category'] \
                           .agg(lambda x: x.value_counts().idxmax()) \
                           .to_dict()
receipts.loc[receipts['capture_category'].isna(), 'capture_category'] = receipts['matched_brand'].map(majority_category)


In [ ]:
# Removes overlapping categories and categories with small sample sizes
item_type_df = receipts.copy()

item_type_df.loc[:, 'category_count'] = item_type_df.groupby(['capture_category'], as_index=False)['receipt_id'].transform('nunique')
item_type_df = item_type_df[item_type_df['category_count'] > 1000].copy()

removed_categories = ['Grocery|Snacks', 'Grocery & Gourmet Food', 'Grocery|Breakfast|Bars|Nutritional Bars', 
                      'Baby', 'Arts, Crafts & Sewing', 'Health & Medicine|Medicine|Cough, Cold & Flu Medicine']
item_type_df = item_type_df[~item_type_df['capture_category'].isin(removed_categories)].copy()

# Convert receipt_purchase_date to datetime and extract month
item_type_df['receipt_purchase_date'] = pd.to_datetime(item_type_df['receipt_purchase_date'], errors='coerce', dayfirst=True)
item_type_df.loc[:, 'month'] = item_type_df['receipt_purchase_date'].dt.to_period('M')

# Group by capture_category and month, counting the number of sales (receipt_id)
monthly_sales = item_type_df.groupby(['capture_category', 'month'])['receipt_id'].nunique().reset_index()

# Rename 'receipt_id' to 'receipt_count' in the monthly_sales DataFrame
monthly_sales.rename(columns={'receipt_id': 'receipt_count'}, inplace=True)

categories = monthly_sales['capture_category'].unique()

In [ ]:
# Initialize dictionaries to store results
trend_line_coefficients = {}
avg_sales_growth_6m_vs_6m = {}
avg_sales_growth_12m_vs_12m = {}

# Calculate growth rates, trend line coefficients first
for category in categories:
    # Filter the data for each category
    category_data = monthly_sales[monthly_sales['capture_category'] == category]
    
    # Sort by month to ensure correct plotting
    category_data = category_data.sort_values(by='month')
    
    # Remove the first 2 and last 2 months of data due to low volume
    first_two_months = category_data['month'].unique()[:2]
    last_two_months = category_data['month'].unique()[-2:]
    category_data = category_data[~category_data['month'].isin(first_two_months)]  
    category_data = category_data[~category_data['month'].isin(last_two_months)]  
    
    # Calculate average sales growth: previous 6 months vs the 6 months before that
    if len(category_data) >= 12:
        avg_sales_last_6m = category_data['receipt_count'].iloc[-6:].mean()  
        avg_sales_prev_6m = category_data['receipt_count'].iloc[-12:-6].mean()  
        avg_sales_growth_6m_vs_6m[category] = ((avg_sales_last_6m - avg_sales_prev_6m) / avg_sales_prev_6m) * 100  
    else:
        avg_sales_growth_6m_vs_6m[category] = np.nan  
    
    # Calculate average sales growth: previous 12 months vs the 12 months before that
    if len(category_data) >= 24:
        avg_sales_last_12m = category_data['receipt_count'].iloc[-12:].mean()  
        avg_sales_prev_12m = category_data['receipt_count'].iloc[-24:-12].mean()  
        avg_sales_growth_12m_vs_12m[category] = ((avg_sales_last_12m - avg_sales_prev_12m) / avg_sales_prev_12m) * 100  
    else:
        avg_sales_growth_12m_vs_12m[category] = np.nan  
    
    # Calculate trend line coefficients
    if len(category_data) >= 6:  
        X = np.arange(len(category_data)).reshape(-1, 1)  
        y = category_data['receipt_count'].values  

        # Fit a linear regression model
        model = LinearRegression()
        model.fit(X, y)
        slope = model.coef_[0]  
        
        # Scale the trend line coefficient by the average number of sales to normalize it
        average_sales = category_data['receipt_count'].mean()
        scaled_slope = slope / average_sales if average_sales != 0 else 0  
        
        trend_line_coefficients[category] = scaled_slope  
    else:
        trend_line_coefficients[category] = np.nan  

# Sort categories by descending trend line coefficients
sorted_categories = sorted(trend_line_coefficients.keys(), key=lambda x: trend_line_coefficients[x], reverse=True)

# Plot the results in the sorted order
num_categories = len(sorted_categories)
n_cols = 3  # You can adjust this based on your preference
n_rows = int(np.ceil(num_categories / n_cols))  # Calculate rows dynamically based on number of categories

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 5), constrained_layout=True)
axes = axes.flatten()  # Flatten axes array for easy iteration

for i, category in enumerate(sorted_categories):
    # Filter the data for each category
    category_data = monthly_sales[monthly_sales['capture_category'] == category]
    
    # Sort by month to ensure correct plotting
    category_data = category_data.sort_values(by='month')
    
    # Remove the first 2 and last 2 months of data due to low volume
    first_two_months = category_data['month'].unique()[:2]
    last_two_months = category_data['month'].unique()[-2:]
    category_data = category_data[~category_data['month'].isin(first_two_months)]  
    category_data = category_data[~category_data['month'].isin(last_two_months)]  
    
    # Create X-axis labels: only show January and July with the year
    category_data['month_timestamp'] = category_data['month'].dt.to_timestamp()
    months = category_data['month_timestamp'].dt.month  
    years = category_data['month_timestamp'].dt.year  
    
    x_labels_clean = [
        f'Jan\n{year}' if month == 1
        else f'Jul\n{year}' if month == 7
        else ''
        for month, year in zip(months, years)
    ]
    
    # Plot the line chart for the category in the respective subplot
    axes[i].plot(category_data['month'].astype(str), category_data['receipt_count'], marker='o', label='Sales')
    axes[i].set_title(f'{category} Sales')
    axes[i].set_xlabel('Month')
    axes[i].set_ylabel('Number of Sales')
    axes[i].tick_params(axis='x', rotation=45)
    
    # Set the cleaned X-axis labels (only Jan and Jul)
    axes[i].set_xticks(category_data['month'].astype(str))
    axes[i].set_xticklabels(x_labels_clean)
    
    # Add the trend line in red
    if len(category_data) >= 6:  
        X = np.arange(len(category_data)).reshape(-1, 1)  
        y = category_data['receipt_count'].values  

        # Fit the linear regression model again for plotting
        model = LinearRegression()
        model.fit(X, y)
        trend_line = model.predict(X)  
        
        # Plot the trend line in red
        axes[i].plot(category_data['month'].astype(str), trend_line, color='red', linestyle='--', label='Trend Line')
    
    axes[i].grid(True)
    axes[i].legend()

# Remove any empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Show all plots
plt.show()

# Convert growth rates and trend line coefficients to DataFrame
data_df = pd.DataFrame({
    'Category': trend_line_coefficients.keys(),
    '6 Month Sales Growth (Previous 6m vs 6m Before) (%)': avg_sales_growth_6m_vs_6m.values(),
    '12 Month Sales Growth (Previous 12m vs 12m Before) (%)': avg_sales_growth_12m_vs_12m.values(),
    'Trend Line Coefficient (Scaled)': trend_line_coefficients.values()  
})


In [ ]:
pd.set_option('display.max_colwidth', None)       # Show full column content
data_df.sort_values(['Trend Line Coefficient (Scaled)'], ascending = False)

# 4. Retention 

### Methodology:
- Reassigning Matched Brand Categories: Initially, I was randomly assigning a single brand to a row when multiple brand matches existed. However, for analyzing retention rates, it makes more sense to associate all relevant brands with a given item description. For example, if both 'Chips Ahoy!' and 'Oreo' are frequently purchased together under the same item description, I now create two separate rows, one for each brand. This approach captures customer behavior more accurately by reflecting all associated brands.
- Defining an Item: I define an item as a unique combination of brand and category. This distinction ensures that variations within a brand (e.g., different categories of a brand's products) and similar items across different brands (e.g., two different brands of cookies) are treated separately in the analysis.
- Top Item Selection: The data is grouped by brand and category, and I filter to retain only the top 10 most purchased items based on transaction volume. This ensures the analysis focuses on the most significant items for each brand.
- Retention Calculation: Retention is calculated as the percentage of customers who repurchase the same item at least once in the subsequent 1-month, 6-month, and 12-month periods. To ensure accurate tracking of repeat purchases, only the first time a customer buys an item is considered as part of the dataset.

In [ ]:
# Define the brand variations dictionary (subset of the brands)
brand_variations = {
    'CHIPS AHOY!': ['chips ahoy', 'chips ahoy!'],  # Handle both with and without exclamation mark
    'SOUR PATCH KIDS': ['sour patch kids', 'sour patch'],  # Handle partial match
    'MAYNARDS BASSETT\'S': ['maynards bassetts', 'swedish fish', 'Bassett\'s'],
    'TATE\'S BAKE SHOP': ['tate\'s', 'tate\'s bake shop', 'tate\'s natural miracle', 'suzanne tate\'s nature series']
}

# Function to build regex with word boundaries for each brand
def build_word_boundary_regex(brand):
    return re.compile(rf'(^|\s|\W){re.escape(brand)}(\s|\W|$)', re.IGNORECASE)

# Compile the regular expressions for brands in the brands list (without variations)
brand_patterns = {brand: build_word_boundary_regex(brand) for brand in brands}

# Add patterns for the brands with variations from brand_variations
variation_patterns = {brand: [build_word_boundary_regex(variation) for variation in variations] for brand, variations in brand_variations.items()}

# Try to match based on the 'capture_brand' column
def match_brand_from_capture(row):
    matched_brands = []
    if pd.isnull(row['capture_brand']):
        return matched_brands
    for brand, pattern in brand_patterns.items():
        if pattern.search(row['capture_brand']):
            matched_brands.append(brand)
    return matched_brands

# Match based on the 'item_description' column
def match_brand_from_description(row):
    matched_brands = []
    for brand, pattern in brand_patterns.items():
        if pattern.search(row['item_description']):
            matched_brands.append(brand)
    return matched_brands

# Match based on the brand variations
def match_brand_from_variations(row):
    matched_brands = []
    for brand, patterns in variation_patterns.items():
        for pattern in patterns:
            if pattern.search(row['item_description']):
                matched_brands.append(brand)
                break  # Break to avoid appending the same brand multiple times for different variations
    return matched_brands

# Combine all matching functions to get a list of matched brands
def find_all_matches(row):
    matched_brands = set()  # Use a set to avoid duplicates
    
    # Try capture_brand column first
    matched_brands.update(match_brand_from_capture(row))
    
    # Try item_description column next
    matched_brands.update(match_brand_from_description(row))
    
    # Try variations based on item_description
    matched_brands.update(match_brand_from_variations(row))
    
    return list(matched_brands)  # Convert back to a list

# Apply the matching function to each row and store the result in 'matched_brand' as a list
receipts['matched_brand'] = receipts.apply(find_all_matches, axis=1)

# Explode the 'matched_brand' list into multiple rows
receipts_exploded = receipts.explode('matched_brand')

# Filter to include only rows where a brand match was found
receipts_exploded = receipts_exploded.dropna(subset=['matched_brand'])

# Create the final DataFrame with user_id and matched_brand columns
final_df = receipts_exploded[['user_id', 'matched_brand', 'capture_category', 'receipt_purchase_date']].reset_index(drop=True)

In [ ]:
item_type_df = final_df
item_type_df['item_count'] = item_type_df.groupby(['matched_brand', 'capture_category'], as_index = False)['user_id'].transform('nunique')
item_type_df['item_count_rank'] = item_type_df['item_count'].rank(method='dense', ascending=False)
item_type_df = item_type_df[item_type_df['item_count_rank'] < 11].copy()
item_type_df['receipt_purchase_date'] = pd.to_datetime(item_type_df['receipt_purchase_date'], errors='coerce', dayfirst=True)

# Finds customers first purchase of the item
item_type_df['first_purchase'] = item_type_df.groupby(['user_id', 'matched_brand', 'capture_category'])['receipt_purchase_date'].rank(method='first')

item_type_df['purchase_year_month'] = item_type_df['receipt_purchase_date'].dt.to_period('M')

# Filter to only keep the first purchase of each item per user 
first_purchases_df = item_type_df[item_type_df['first_purchase'] == 1].copy()

retention_data_list = []

# Loop over each unique matched_brand and capture_category combination
for (brand, category), group in first_purchases_df.groupby(['matched_brand', 'capture_category']):
    
    retention_data = {'matched_brand': brand, 'capture_category': category}

    group['next_month'] = group['purchase_year_month'] + 1

    # Next-Month Retention Check - check if a purchase occurred in the next month
    retention_df_next_month = pd.merge(
        group,
        item_type_df[['user_id', 'matched_brand', 'capture_category', 'purchase_year_month']], 
        how='left',
        left_on=['user_id', 'matched_brand', 'capture_category', 'next_month'],
        right_on=['user_id', 'matched_brand', 'capture_category', 'purchase_year_month'],
        suffixes=('', '_next_month')
    )
    retention_df_next_month['retained_next_month'] = retention_df_next_month['purchase_year_month_next_month'].notna()

    retention_rate_next_month = retention_df_next_month['retained_next_month'].mean() * 100  
    retention_data['next_month_retention_rate (%)'] = round(retention_rate_next_month, 2)

    # 6-Month Retention Check - checks if a purchase occurs in any of the next 6 months
    group['next_6_month_start'] = group['purchase_year_month'] + 1
    group['next_6_month_end'] = group['purchase_year_month'] + 6

    retention_df_6m = pd.merge(
        group,
        item_type_df[['user_id', 'matched_brand', 'capture_category', 'purchase_year_month']],  # Necessary columns for the join
        how='left',
        left_on=['user_id', 'matched_brand', 'capture_category'],
        right_on=['user_id', 'matched_brand', 'capture_category'],
        suffixes=('', '_next')
    )

    retention_df_6m['within_6_months'] = retention_df_6m.apply(
        lambda row: row['next_6_month_start'] <= row['purchase_year_month_next'] <= row['next_6_month_end'],
        axis=1
    )

    retention_rate_6m = retention_df_6m['within_6_months'].mean() * 100  # Convert to percentage
    retention_data['6_month_retention_rate (%)'] = round(retention_rate_6m, 2)

    # 12-Month Retention Check - checks if a purchase occurs in any of the next 12 months
    group['next_12_month_start'] = group['purchase_year_month'] + 1
    group['next_12_month_end'] = group['purchase_year_month'] + 12

    retention_df_12m = pd.merge(
        group,
        item_type_df[['user_id', 'matched_brand', 'capture_category', 'purchase_year_month']],  # Necessary columns for the join
        how='left',
        left_on=['user_id', 'matched_brand', 'capture_category'],
        right_on=['user_id', 'matched_brand', 'capture_category'],
        suffixes=('', '_next')
    )

    retention_df_12m['within_12_months'] = retention_df_12m.apply(
        lambda row: row['next_12_month_start'] <= row['purchase_year_month_next'] <= row['next_12_month_end'],
        axis=1
    )

    retention_rate_12m = retention_df_12m['within_12_months'].mean() * 100  # Convert to percentage
    retention_data['12_month_retention_rate (%)'] = round(retention_rate_12m, 2)

    retention_data_list.append(retention_data)

#Convert the list of retention data into a DataFrame
retention_rates_df = pd.DataFrame(retention_data_list)

retention_rates_df.sort_values(['next_month_retention_rate (%)'], ascending = False)

# 5. Brand Overlap

### Methodology

- Calculating Brand Overlaps: For each brand, I calculate the proportion of its customers who also purchase every other brand. For every brand-to-brand combination, two proportions are computed: (1) the proportion of customers who bought brand A that also bought brand B, and (2) the proportion of customers who bought brand B that also bought brand A (note that these proportions are different).
- Balancing Overlap Perspectives: Create a new metric by adding the two proportions together for each brand-to-brand combination, ensuring equal weight is given to both sides of the overlap.
- Filtering for Sample Size: To reduce noise from small sample sizes, filter out all brand combinations where fewer than 100 customers are involved in the overlap.
- Visualizing Brand Overlaps: Create a grid displaying the overlaps between all brands, using the new metric to highlight the strongest relationships.

In [ ]:
overlap_df = final_df[['user_id', 'matched_brand']]
brands = overlap_df['matched_brand'].unique()
overlap_proportions = {}

# Calculate overlap proportions
for brand in brands:
    brand_users = set(overlap_df[overlap_df['matched_brand'] == brand]['user_id'])
    
    overlap_proportions[brand] = {}
    
    for other_brand in brands:
        if brand != other_brand:  
            other_brand_users = set(overlap_df[overlap_df['matched_brand'] == other_brand]['user_id'])
            
            overlap_count = len(brand_users.intersection(other_brand_users))
            
            # Only include if the overlap is at least 100 customers
            if overlap_count >= 100:
                proportion = overlap_count / len(brand_users) if len(brand_users) > 0 else 0
                overlap_proportions[brand][other_brand] = round(proportion, 4) 
            else:
                overlap_proportions[brand][other_brand] = None  # If overlap < 100, set to None

overlap_df = pd.DataFrame(overlap_proportions).T

sum_df = pd.DataFrame(index=overlap_df.index, columns=overlap_df.columns)

# Loop through each brand pair and calculate the sum of percentages
for brand in overlap_df.index:
    for other_brand in overlap_df.columns:
        if brand != other_brand:  # Only compute for different brands
            if overlap_df.at[brand, other_brand] is not None and overlap_df.at[other_brand, brand] is not None:
                # Sum the proportion from brand to other_brand and other_brand to brand
                sum_proportion = overlap_df.at[brand, other_brand] + overlap_df.at[other_brand, brand]
                sum_df.at[brand, other_brand] = round(sum_proportion, 4) 
            else:
                sum_df.at[brand, other_brand] = None  # If either overlap is less than 100, set to None
        else:
            sum_df.at[brand, other_brand] = None  # Leave diagonal (same brand) empty

# Display the new DataFrame with the sum of percentages
sum_df

# 6. Demographic Data


- Bias in Methodology: The first area I’d investigate is whether there’s any bias in my methodology. I removed around 300,000 rows that I couldn’t match to a given brand, and it’s important to understand if those removed rows represent a specific demographic group. If I’m systematically excluding a certain population from my dataset, it could skew my results and findings. To ensure reliability, I’d compare the demographics of the removed rows with the rest of the data to make sure I’m not unintentionally excluding an important group.
- Assessing Future Potential Through Demographic Trends: By comparing demographic data to broader trends in population changes, KKR can gain insights into a brand's future potential. For instance, a brand that resonates with demographics expected to grow or increase in buying power may be more valuable in the long term compared to one tied to shrinking or less dynamic groups.
- Dataset Bias and Representation: It's important to explore potential biases in the dataset, as it only includes sales from Amazon, Amazon Fresh, and Whole Foods, which may not represent the broader population. These platforms could overrepresent certain groups, such as males or rural consumers, potentially limiting the applicability of findings to other sales channels like grocery stores or regional markets. Understanding which groups are over- or underrepresented is crucial for assessing the scope and validity of conclusions.